In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from glob import glob
import rasterio as rio
from rasterio.mask import mask
from rasterio.io import MemoryFile
import pickle
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from src.mslandcover.config import MSTM_PROJ4
from src.mslandcover.utils import raise_if_not_exists
import cv2

In [ ]:
# load the shapefilse with boundaries of the regions

shapefiles = glob(r'data\MS_NAIP_2023\*\*.shp')
gdfs = []
for shapefile in shapefiles:
    gdf = gpd.read_file(shapefile).to_crs(MSTM_PROJ4) # convert to the same projection
    raster_path = glob(os.path.join(os.path.dirname(shapefile), '*_1m.tif'))[0]
    gdf['raster_path'] = raster_path
    gdfs.append(gdf)


raster_boundaries_gdf = gpd.GeoDataFrame(pd.concat(gdfs))[['raster_path', 'geometry']] # only keep the relevant columns
raster_boundaries_gdf = raster_boundaries_gdf.dissolve(by='raster_path').reset_index() # dissolve the geometries to get the boundaries of the raster
raster_boundaries_gdf.to_file('data/regions_boundaries.gpkg', driver='GPKG')

In [3]:
# load the samples parquet
samples = gpd.read_parquet('./data/sampling/samples.par')
raster_boundaries_gdf = gpd.read_file('data/regions_boundaries.gpkg')
# print(raster_boundaries_gdf)

raster_boundaries_gdf['geometry'] = raster_boundaries_gdf.buffer(0).buffer(-20) # fix invalid geometries and shrink the boundaries by 10 meters to avoid cases where the sample intersects the boundary
raster_boundaries_gdf['raster_geometry'] = raster_boundaries_gdf['geometry'] # make a copy of the geometry
# spatial join with the raster boundaries
samples = gpd.sjoin(samples, raster_boundaries_gdf, predicate='intersects', how='left')
samples['intersection_area'] = samples.apply(lambda x: x['geometry'].intersection(x['raster_geometry']).area, axis=1)

In [ ]:

# # drop duplicates - keep those with the largest intersection area
# samples = samples.sort_values('intersection_area', ascending=False)
# # samples = samples[~samples.index.duplicated(keep='first')]
# samples['intersection_percentage'] = samples['intersection_area'] / samples['geometry'].area

# samples = samples[samples['intersection_percentage'] == 1]
# samples.groupby('split').count()

,geometry,hist_vector,hist_vector_scaled,hist_vector_pca,cluster,index_right,raster_path,raster_geometry,intersection_area,intersection_percentage
split,,,,,,,,,,
pretrain,377919,377919,377919,377919,377919,377919,377919,377919,377919,377919
test,250,250,250,250,250,250,250,250,250,250
train,500,500,500,500,500,500,500,500,500,500
val,250,250,250,250,250,250,250,250,250,250


In [ ]:
def extract_mask(sample, raster_dataset):
    
    out_image, out_transform = mask(raster_dataset, [sample['geometry']], crop=True, all_touched=True, )
    
    out_meta = raster_dataset.meta.copy()

    if out_image.shape[1] > 256 or out_image.shape[2] > 256:
        # crop the image to 256x256
        out_image = out_image[:, :256, :256]
    
    filename = str(sample.name)
    if out_image.shape != (3, 256, 256):\
        filename += '_BADSHAPE'
    
    nd_values = [raster_dataset.nodata] * 3
    if np.any(out_image.transpose(1, 2, 0).reshape(-1, 3) == nd_values).any():
        return
    
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform,
    })
    
    out_path = os.path.join('data', 'splits', sample['split'], filename + '.tif')
    with rio.open(out_path, 'w', **out_meta) as dst:
        dst.write(out_image)

def extract_raster(samples_group):
    raster_path = samples_group[0]
    with MemoryFile(open(raster_path, 'rb').read()) as memfile:
        with rio.open(memfile) as raster_dataset:
            samples_group[1].apply(lambda x: extract_mask(x, raster_dataset), axis=1)

sub_samples = samples[samples['split'].isin(['train', 'test', 'val'])]
n_threads = os.cpu_count()
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(sub_samples.groupby('raster_path'))))

In [ ]:
dummy_img = np.zeros((3, 256, 256), dtype=np.uint8)
dummy_nd_value = [0, 0, 0]

# all three bands will have 0 if the pixel is nodata (i.e., [0, 0, 0])
def does_image_contain_nd(img):
    return img.transpose(1, 2, 0).reshape(-1, 3).all(axis=1).all()
    return (img[0] == 0).any() and (img[1] == 0).any() and (img[2] == 0).any()

print((dummy_img.transpose(1, 2, 0).reshape(-1, 3) == [0, 0, 0]).any())

True


<!-- ## Sampling Points for Accuracy Assesment -->

In [ ]:
# criteria for selecting points for accuracy assessment
# 1. points are stratified by NLCD Level I class
# 3. points are within the state boundary
# 4. points are note within the same grid cell as a sample patch (pretraining, train, val, or test)
# 5. no more than 1 point per grid cell

# load NLCD level I data
with rio.open('./data/sampling/NLCD/NLCD_MS_level_1.tif') as src:
    nlcd_level_I_data = src.read(1)
    nlcd_level_I_meta = src.meta
    
    row_col_to_xy = src.xy
    xy_to_row_col = src.index

# get distirbution of NLCD Level I classes
_, class_counts = np.unique(nlcd_level_I_data, return_counts=True)
class_probs = class_counts / class_counts.sum()
# first class is nodoata, so remove it
class_probs = class_probs[1:] ** 0.5 # square root to increase the number of points for classes with lower probabilities

min_class = class_probs.min()

n_points_per_class = np.round(200 * class_probs / min_class).astype(int)
print(n_points_per_class, n_points_per_class.sum())

def sample_points(class_index, points_to_sample, nlcd_level_I_data, ms_polygon, remaining_candidate_patches_gdf):
    
    running_class_points = 0
    sample_points = []
    while running_class_points < points_to_sample:
        
        row = np.random.randint(0, nlcd_level_I_data.shape[0])
        col = np.random.randint(0, nlcd_level_I_data.shape[1])
        
        # check to make sure point is of the correct class
        if nlcd_level_I_data[row, col] != class_index:
            continue
        
        # transform row, col to x, y
        x, y = row_col_to_xy(row, col)
        point = Point(x, y)
        
        # check if point is within state boundary
        if not ms_polygon.contains(point):
            continue
        
        # check if point is in an eligible grid cell
        intersecting_grid_cell = remaining_candidate_patches_gdf.intersects(point)
        if intersecting_grid_cell.any():
            continue
        
        # remove grid cell that intersects point
        remaining_candidate_patches_gdf = remaining_candidate_patches_gdf.drop(remaining_candidate_patches_gdf[intersecting_grid_cell].index)
        # remaining_candidate_patches_gdf = remaining_candidate_patches_gdf[~intersecting_grid_cell]
        
        sample_points.append({
            'class': class_index,
            'geometry': point,
        })
        
        running_class_points += 1
        
        if running_class_points % 10 == 0:
            print(f'Class {class_index}: {running_class_points}/{points_to_sample} points sampled', end='\r')

    print(f'Class {class_index}: {running_class_points} points sampled')
    return sample_points

for i, n in enumerate(n_points_per_class):
    sample_points_futures = sample_points(i+1, n, nlcd_level_I_data, ms_polygon, remaining_candidate_patches_gdf)

sample_points = list(sample_points_futures)
with open('./data/sampling/sample_points.pkl', 'wb') as f:
    pickle.dump(sample_points, f)